# 03 · Análisis descriptivo y visualización

Análisis de los datos limpios (`data/processed/bank_customers_clean.csv`) para responder a la pregunta de negocio: **¿qué características tienen los clientes que contratan el depósito a plazo?**

In [1]:
# Cargar los datos limpios
import pandas as pd

ruta_datos = "../data/processed/bank_customers_clean.csv"

df = pd.read_csv(ruta_datos, parse_dates=["date", "Dt_Customer"])

print(df.shape)
print(df.dtypes)

(43000, 28)
age                           int64
job                             str
marital                         str
education                       str
default                     float64
housing                     float64
loan                        float64
contact                         str
duration                      int64
campaign                      int64
pdays                         int64
previous                      int64
poutcome                        str
emp.var.rate                float64
cons.price.idx              float64
cons.conf.idx               float64
euribor3m                   float64
nr.employed                 float64
y                             int64
date                 datetime64[us]
id_                             str
contact_month               float64
contact_year                float64
Income                        int64
Kidhome                       int64
Teenhome                      int64
Dt_Customer          datetime64[us]
NumWebVisitsMont

In [2]:
# El CSV no guarda el tipo Int64: lo recupero en las columnas con nulos
cols_enteras = ["default", "housing", "loan", "contact_month", "contact_year"]

df[cols_enteras] = df[cols_enteras].astype("Int64")

print(df[cols_enteras].dtypes)

default          Int64
housing          Int64
loan             Int64
contact_month    Int64
contact_year     Int64
dtype: object


In [3]:
# Columnas numéricas que describen al cliente y a la llamada
cols_numericas = ["age", "duration", "campaign", "pdays", "previous",
                  "Income", "Kidhome", "Teenhome", "NumWebVisitsMonth"]

resumen = df[cols_numericas].describe().T
resumen["median"] = df[cols_numericas].median()

print(resumen[["mean", "median", "std", "min", "max"]].round(1))

                      mean   median      std     min       max
age                   39.7     38.0      9.8    17.0      98.0
duration             257.7    179.0    258.7     0.0    4918.0
campaign               2.6      2.0      2.8     1.0      56.0
pdays                962.3    999.0    187.3     0.0     999.0
previous               0.2      0.0      0.5     0.0       7.0
Income             93241.2  93050.5  50498.3  5841.0  180802.0
Kidhome                1.0      1.0      0.8     0.0       2.0
Teenhome               1.0      1.0      0.8     0.0       2.0
NumWebVisitsMonth     16.6     17.0      9.2     1.0      32.0


In [4]:
# pdays = 999 es un código ("sin días registrados"), no un número de días
sin_dias = df["pdays"] == 999

print(f"Clientes con pdays = 999: {sin_dias.sum()} ({sin_dias.mean() * 100:.1f} %)")
print(df.loc[~sin_dias, "pdays"].describe().round(1))

Clientes con pdays = 999: 41412 (96.3 %)
count    1588.0
mean        6.1
std         3.9
min         0.0
25%         3.0
50%         6.0
75%         7.0
max        27.0
Name: pdays, dtype: float64


In [5]:
# Comprobar el significado del 999 cruzándolo con el resultado de la campaña anterior
print(pd.crosstab(sin_dias, df["poutcome"]))

poutcome  failure  nonexistent  success
pdays                                  
False         152            0     1436
True         4309        37103        0


In [6]:
# ¿Contratan más los clientes con contacto reciente?
grupo = sin_dias.map({True: "sin días (999)", False: "con días registrados"})
print(df.groupby(grupo)["y"].mean().round(3))

pdays
con días registrados    0.640
sin días (999)          0.092
Name: y, dtype: float64


In [7]:
# Comparar las variables numéricas entre quien contrata (1) y quien no (0)
cols_comparar = ["age", "duration", "campaign", "previous",
                 "Income", "Kidhome", "Teenhome", "NumWebVisitsMonth"]

print(df.groupby("y")[cols_comparar].mean().round(1).T)

y                        0        1
age                   39.6     40.5
duration             220.4    551.6
campaign               2.6      2.1
previous               0.1      0.5
Income             93324.6  92584.3
Kidhome                1.0      1.0
Teenhome               1.0      1.0
NumWebVisitsMonth     16.6     16.6


In [8]:
def tasa_exito(datos, columna):
    """Devuelve el número de clientes y la tasa de éxito (%) de cada categoría."""
    tabla = datos.groupby(columna)["y"].agg(["count", "mean"])
    tabla["mean"] = (tabla["mean"] * 100).round(1)
    tabla = tabla.rename(columns={"count": "clientes", "mean": "tasa_exito_%"})
    return tabla.sort_values("tasa_exito_%", ascending=False)

In [9]:
# Tasa de éxito por cada variable categórica
cols_categoricas = ["job", "marital", "education", "contact", "poutcome"]

for col in cols_categoricas:
    print(tasa_exito(df, col))
    print("-" * 40)

               clientes  tasa_exito_%
job                                  
student             903          31.3
retired            1790          25.2
unemployed         1063          14.4
admin             10873          13.0
unknown             345          11.3
management         3050          11.2
technician         7026          10.8
self-employed      1489          10.8
housemaid          1123           9.9
entrepreneur       1522           8.3
services           4162           8.1
blue-collar        9654           6.9
----------------------------------------
          clientes  tasa_exito_%
marital                         
unknown         85          15.3
single       12105          13.9
married      25999          10.2
divorced      4811          10.2
----------------------------------------
                     clientes  tasa_exito_%
education                                  
illiterate                 18          22.2
unknown                  1807          14.4
university_d

           clientes  tasa_exito_%
contact                          
cellular      27396          14.7
telephone     15604           5.2
----------------------------------------
             clientes  tasa_exito_%
poutcome                           
success          1436          65.3
failure          4461          14.2
nonexistent     37103           8.8
----------------------------------------


## Hallazgos

- **Tasa de éxito global: 11,3 %** (4.844 de 43.000 llamadas).
- **pdays = 999 es un código**, no días: lo tiene el 96,3 % de los clientes. Su media (962) no tiene sentido; los días reales van de 0 a 27 (mediana 6). En el dataset original (UCI) significa "no contactado", pero aquí 4.309 clientes con 999 tienen poutcome = failure, así que se interpreta como "sin días registrados".
- **Clientes con contacto reciente** (pdays ≠ 999): tasa de éxito del **64 %**, frente al 9 % del resto. El 90 % de ellos ya contrató en la campaña anterior.
- **duration**: las llamadas con éxito duran 2,5 veces más (552 s frente a 220 s), pero solo se conoce al terminar la llamada: no sirve para elegir a quién llamar.
- **campaign**: quien contrata necesitó menos llamadas (2,1 frente a 2,6). Insistir no mejora el resultado.
- **Datos de cliente** (ingresos, hijos, visitas web): prácticamente iguales entre quien contrata y quien no.
- **Profesión**: estudiantes (31,3 %) y jubilados (25,2 %) son los que más contratan; blue-collar (6,9 %) el que menos, siendo además el segundo grupo más llamado (9.654). Volumen de llamadas ≠ eficacia.
- **Canal**: por móvil se contrata casi 3 veces más que por fijo (14,7 % frente a 5,2 %).
- **Campaña anterior**: quien contrató antes (poutcome = success) repite en el 65,3 % de los casos.
- **Educación y estado civil**: universitarios (13,7 %) y solteros (13,9 %) algo por encima de la media. Se descartan los grupos muy pequeños (illiterate: 18 clientes; marital unknown: 85) por no ser representativos.